# Modelado

---

**Dataset:** [Nombre del dataset]

---

## Que hacemos en el Modelado?

El Feature Engineering nos entrego `X_train`, `X_test`, `y_train`, `y_test` listos.
Aqui construimos el Pipeline de transformacion + modelo, lo entrenamos y generamos predicciones.

| Paso | Descripcion |
|---|---|
| 1 | Importar librerias y cargar objetos del FE |
| 2 | Definir columnas del Pipeline |
| 3 | Construir Pipeline (sub-pipelines + ColumnTransformer + modelo) |
| 4 | Entrenar el Pipeline y generar predicciones |
| 5 | Exportar el Pipeline para produccion |
| 6 | Resumen del Modelado |

> **Regla de oro del Pipeline:** `fit` SOLO con datos de train.
> `transform` se aplica a train y test por igual. Esto previene el Data Leakage.

---


---
## Paso 1 — Importar librerias

| Libreria / Modulo | Para que la usamos |
|---|---|
| `pandas` / `numpy` | Manipulacion de datos |
| `sklearn.pipeline` | Construir el Pipeline reproducible |
| `sklearn.compose` | Aplicar transformaciones por tipo de columna |
| `sklearn.impute` | Imputacion dentro del Pipeline |
| `sklearn.preprocessing` | Escalado y codificacion dentro del Pipeline |
| `sklearn.linear_model` | Modelos de regresion |
| `sklearn.ensemble` | Modelos de ensamble (RandomForest, GradientBoosting) |
| `sklearn.model_selection` | train_test_split, cross_val_score |
| `joblib` | Exportar el Pipeline entrenado |

---


In [1]:
# ── Librerias ────────────────────────────────────────────────────
# ── Manipulacion de datos ────────────────────────────────────────────────────
import pandas as pd
import numpy as np

# ── Visualizacion ────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

# ── Pipeline y ColumnTransformer ─────────────────────────────────────────────
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# ── Preprocesamiento dentro del Pipeline ─────────────────────────────────────
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (StandardScaler, MinMaxScaler, RobustScaler,
                                   OneHotEncoder, FunctionTransformer)

# ── Modelos ───────────────────────────────────────────────────────────────────
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
# from xgboost import XGBRegressor  # descomenta si tienes xgboost instalado

# ── Evaluacion ────────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# ── Exportacion ───────────────────────────────────────────────────────────────
import joblib, os

# ── Configuracion ─────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print('Librerias cargadas correctamente')


Librerias cargadas correctamente


---
## Paso 2 — Cargar objetos del Feature Engineering

Los objetos `X_train`, `X_test`, `y_train`, `y_test`, `num_pipeline_cols`,
`cat_pipeline_cols` y `date_pipeline_cols` deben estar en memoria
desde el Notebook de Feature Engineering.

**Alternativa:** si los guardaste como archivos, cargalos aqui.

---


In [2]:
# ── Configuracion ─────────────────────────────────────
TARGET_COL = 'Price_log'   # <-- el nombre de tu target con sufijo '_log' si lo transformaste con log1p

In [3]:
# ── Cargar datos del Feature Engineering ─────────────────────────────────────

train = pd.read_parquet("work_lineal/train_final.parquet")
test  = pd.read_parquet("work_lineal/test_final.parquet")

X_train = train.drop(columns=[TARGET_COL])
y_train = train[TARGET_COL]

X_test = test.drop(columns=[TARGET_COL])
y_test = test[TARGET_COL]

# Clasificar columnas para el ColumnTransformer
num_pipeline_cols  = X_train.select_dtypes(include='number').columns.tolist()
cat_pipeline_cols  = X_train.select_dtypes(include='object').columns.tolist()
date_pipeline_cols = X_train.select_dtypes(include='datetime').columns.tolist()

print(f'X_train: {X_train.shape}  |  X_test: {X_test.shape}')
print(f'Numericas: {len(num_pipeline_cols)}  |  Categoricas: {len(cat_pipeline_cols)}  |  Fechas: {len(date_pipeline_cols)}')

X_train: (10853, 20)  |  X_test: (2714, 20)
Numericas: 20  |  Categoricas: 0  |  Fechas: 0


In [4]:
# ── Verificar que los objetos del FE estan disponibles ──────────────────────
# Si estan en memoria (desde el Notebook de FE), este bloque lo confirma.
# Si no, usa las opciones de carga comentadas abajo.

try:
    print('Objetos del FE disponibles en memoria:')
    print(f'   X_train: {X_train.shape[0]:,} filas x {X_train.shape[1]} columnas')
    print(f'   X_test:  {X_test.shape[0]:,} filas x {X_test.shape[1]} columnas')
    print(f'   y_train: {y_train.shape}')
    print(f'   y_test:  {y_test.shape}')
    print(f'   num_pipeline_cols  ({len(num_pipeline_cols)}): {num_pipeline_cols[:5]}...')
    print(f'   cat_pipeline_cols  ({len(cat_pipeline_cols)}): {cat_pipeline_cols}')
    print(f'   date_pipeline_cols ({len(date_pipeline_cols)}): {date_pipeline_cols}')
except NameError as e:
    print(f'AVISO: objeto no encontrado: {e}')
    print('Ejecuta primero el Notebook de Feature Engineering.')
    print('O carga los archivos si los guardaste:')
    print('   X_train = pd.read_parquet("work_lineal/X_train.parquet")')
    print('   pipeline_final = joblib.load("output_lineal/pipeline_modelo.pkl")')


Objetos del FE disponibles en memoria:
   X_train: 10,853 filas x 20 columnas
   X_test:  2,714 filas x 20 columnas
   y_train: (10853,)
   y_test:  (2714,)
   num_pipeline_cols  (20): ['Rooms', 'Distance', 'Bathroom', 'Car', 'Landsize']...
   cat_pipeline_cols  (0): []
   date_pipeline_cols (0): []


---
## Paso 3 — Construir el Pipeline

El Pipeline encadena preprocesamiento + modelo en un objeto unico.
La clave es el **ColumnTransformer**: aplica un pipeline diferente
a cada tipo de columna de forma simultanea.

```
X_train --> ColumnTransformer --> [num: imputa + escala   ]
                               --> [cat: imputa + OHE      ]
                               --> [dat: convierte + escala]
         --> X_transformado --> Modelo.fit()
```

---


In [5]:
# DECISION POINT ── 3a. Sub-pipelines por tipo de columna ────────────────────────────────────

# ── KNNImputer (opcional) ─────────────────────────────────────────────────────
# Usar cuando los nulos tienen patron y SimpleImputer no es suficiente.
# n_neighbors=5: usa los 5 vecinos mas similares para estimar el valor faltante.
# DECISION POINT: descomenta si quieres usar KNN en lugar de SimpleImputer

# from sklearn.impute import KNNImputer
# pipeline_numericas = Pipeline(steps=[
#     ('imputar', KNNImputer(n_neighbors=5, weights='uniform')),
#     ('escalar', RobustScaler())
# ])

# ── IterativeImputer (opcional) ───────────────────────────────────────────────
# Usar cuando los nulos tienen relacion con otras variables del dataset.
# Entrena un modelo interno por cada columna con nulos para predecir su valor.
# DECISION POINT: descomenta si quieres usar IterativeImputer

# from sklearn.experimental import enable_iterative_imputer
# from sklearn.impute import IterativeImputer
# pipeline_numericas = Pipeline(steps=[
#     ('imputar', IterativeImputer(max_iter=10, random_state=42)),
#     ('escalar', RobustScaler())
# ])

# ── Target Encoding para alta cardinalidad (opcional) ─────────────────────────
# Usar cuando tienes columnas categoricas con muchas categorias (>15)
# que discriminan bien el target (Suburb, CouncilArea, etc.)
# Se hace aqui dentro del Pipeline para evitar Data Leakage.
# DECISION POINT: descomenta y define cols_target_enc si lo necesitas

# def datetime_a_timestamp(X):
#     media_global = None  # se aprende en fit()
#     mapping_ = {}
#
# class TargetEncoderCV(BaseEstimator, TransformerMixin):
#     def __init__(self, suavizado=10):
#         self.suavizado = suavizado
#
#     def fit(self, X, y):
#         self.media_global_ = y.mean()
#         self.mapping_ = {}
#         for col in X.columns:
#             agg = pd.DataFrame({'target': y}).groupby(X[col])['target'].agg(['mean','count'])
#             lam = agg['count'] / (agg['count'] + self.suavizado)
#             agg['encoded'] = lam * agg['mean'] + (1 - lam) * self.media_global_
#             self.mapping_[col] = agg['encoded'].to_dict()
#         return self
#
#     def transform(self, X):
#         X_out = X.copy()
#         for col in X.columns:
#             X_out[col] = X[col].map(self.mapping_[col]).fillna(self.media_global_)
#         return X_out.astype(float)
#
# cols_target_enc = ['Suburb', 'CouncilArea']  # <-- define tus columnas
# pipeline_target_enc = Pipeline(steps=[
#     ('imputar', SimpleImputer(strategy='most_frequent')),
#     ('te', TargetEncoderCV(suavizado=10))
# ])

# ── Pipeline numericas (activo por defecto) ───────────────────────────────────
pipeline_numericas = Pipeline(steps=[
    ('imputar', SimpleImputer(strategy='median')),
    ('escalar', RobustScaler())
])

# ── Pipeline categoricas (activo por defecto) ─────────────────────────────────
pipeline_categoricas = Pipeline(steps=[
    ('imputar',   SimpleImputer(strategy='most_frequent')),
    ('codificar', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# ── Pipeline fechas (activo por defecto) ──────────────────────────────────────
def datetime_a_timestamp(X):
    return X.apply(lambda col: col.astype('int64') // 10**9).astype(float)

pipeline_fechas = Pipeline(steps=[
    ('a_numero', FunctionTransformer(datetime_a_timestamp)),
    ('escalar',  StandardScaler())
])

print('Sub-pipelines definidos correctamente')

Sub-pipelines definidos correctamente


In [6]:
# DECISION POINT ── 3b. ColumnTransformer: aplicar cada pipeline a sus columnas ──────────────
# ColumnTransformer aplica en paralelo cada sub-pipeline a su conjunto de columnas.
# remainder='drop': descarta columnas no incluidas en ningun transformer.

transformers = [('num', pipeline_numericas, num_pipeline_cols)]

if cat_pipeline_cols:
    transformers.append(('cat', pipeline_categoricas, cat_pipeline_cols))

if date_pipeline_cols:
    transformers.append(('fecha', pipeline_fechas, date_pipeline_cols))

# DECISION POINT: descomenta si activaste Target Encoding en 3a
# transformers.append(('te', pipeline_target_enc, cols_target_enc))

preprocessor = ColumnTransformer(
    transformers=transformers,
    remainder='passthrough'
)

print(f'ColumnTransformer: {len(transformers)} transformadores definidos')


ColumnTransformer: 1 transformadores definidos


In [7]:
# DECISION POINT ── 3c. Pipeline final: preprocesamiento + modelo ACA SE DEFINE EL MODELO A ENTRENAR ────────────────────────────
# DECISION POINT: elige el modelo. Empieza con LinearRegression como baseline.
# Cambia solo esta linea para probar otros modelos.
#
# Modelos disponibles:
# modelo_elegido = LinearRegression()                            # Baseline interpretable
# modelo_elegido = Ridge(alpha=1.0)                              # Regularizacion L2
# modelo_elegido = Lasso(alpha=0.1)                              # Regularizacion L1
# modelo_elegido = RandomForestRegressor(n_estimators=100, random_state=42)
# modelo_elegido = GradientBoostingRegressor(n_estimators=100, random_state=42)

modelo_elegido = GradientBoostingRegressor(n_estimators=100, random_state=42)   # <-- CAMBIA POR TU MODELO

pipeline_final = Pipeline(steps=[
    ('preprocesamiento', preprocessor),
    ('modelo', modelo_elegido)
])

print('Pipeline final construido:')
print(f'   Preprocesamiento : ColumnTransformer ({len(transformers)} grupos)')
print(f'   Modelo           : {type(modelo_elegido).__name__}')
print()
print(pipeline_final)


Pipeline final construido:
   Preprocesamiento : ColumnTransformer (1 grupos)
   Modelo           : GradientBoostingRegressor

Pipeline(steps=[('preprocesamiento',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num',
                                                  Pipeline(steps=[('imputar',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('escalar',
                                                                   RobustScaler())]),
                                                  ['Rooms', 'Distance',
                                                   'Bathroom', 'Car',
                                                   'Landsize', 'Propertycount',
                                                   'CouncilArea_era_nulo',
                                                   '+3_rooms', 'Date_ano',
   

---
## Paso 4 — Entrenar el Pipeline

El metodo `.fit()` hace todo en una sola llamada:
1. Aprende medianas, modas, categorias **SOLO con X_train**
2. Transforma X_train con esos parametros
3. Entrena el modelo con los datos transformados

`.predict(X_test)` aplica las mismas transformaciones usando los parametros del train.
No re-aprende nada del test. Eso es lo que previene el Data Leakage.

---


In [8]:
# ── 4a. Entrenar el Pipeline ──────────────────────────────────────────────────

pipeline_final.fit(X_train, y_train)
print('Pipeline entrenado exitosamente')


Pipeline entrenado exitosamente


In [9]:
# ── 4b. Generar predicciones ─────────────────────────────────────────────────

y_pred       = pipeline_final.predict(X_test)    # predicciones en test
y_pred_train = pipeline_final.predict(X_train)   # predicciones en train

print(f'Predicciones generadas:')
print(f'   y_pred (test):  {y_pred.shape}')
print(f'   y_pred (train): {y_pred_train.shape}')


Predicciones generadas:
   y_pred (test):  (2714,)
   y_pred (train): (10853,)


In [10]:
# ── 4c. Inspeccion de las primeras predicciones ──────────────────────────────
# Comparamos los primeros valores reales vs predichos para una sanidad check.
# Si el target fue transformado (log), las predicciones estan en la misma escala.
# Aplica la transformacion inversa para interpretar en escala original:
#   Si aplicaste log1p -> np.expm1(y_pred)
#   Si aplicaste sqrt  -> y_pred**2

pred_df = pd.DataFrame({
    'Real': y_test.values[:10],
    'Predicho': y_pred[:10],
    'Diferencia': y_test.values[:10] - y_pred[:10]
})
print('Primeras 10 predicciones (escala del target transformado):')
display(pred_df.round(4))

if 'log' in TARGET_COL:
    pred_original = pd.DataFrame({
        'Real (original)':     np.expm1(y_test.values[:10]),
        'Predicho (original)': np.expm1(y_pred[:10]),
    })
    print('\nPrimeras 10 predicciones en escala original (expm1):')
    display(pred_original.round(0))


Primeras 10 predicciones (escala del target transformado):


,Real,Predicho,Diferencia
0,12.5707,12.7339,-0.1632
1,13.4939,13.6585,-0.1645
2,14.2210,13.8268,0.3942
3,13.1616,13.2094,-0.0478
4,14.4269,13.9548,0.4722
5,13.5371,13.7080,-0.1709
6,14.5598,14.7345,-0.1747
7,14.4833,14.1863,0.2970
8,14.3432,14.1142,0.2290
9,13.3375,13.3416,-0.0042



Primeras 10 predicciones en escala original (expm1):


,Real (original),Predicho (original)
0,288000.0000,339054.0000
1,725000.0000,854657.0000
2,1500000.0000,1011375.0000
3,520000.0000,545467.0000
4,1843000.0000,1149401.0000
5,757000.0000,898060.0000
6,2105000.0000,2506723.0000
7,1950000.0000,1448901.0000
8,1695000.0000,1348047.0000
9,620000.0000,622587.0000


---
## Paso 5 — Resumen del Modelado

---


In [11]:
# ── Resumen del Modelado ─────────────────────────────────────────────────────

sep  = '-' * 70
sep2 = '=' * 70

print(sep2)
print('  RESUMEN DE MODELADO')
print(f'  Modelo: {type(pipeline_final.named_steps["modelo"]).__name__}')
print(sep2)

print(f'\n1. PIPELINE')
print(sep)
for nombre, transformador in pipeline_final.named_steps.items():
    print(f'   {nombre:<20}: {type(transformador).__name__}')

print(f'\n2. DATOS')
print(sep)
print(f'   X_train: {X_train.shape[0]:,} filas x {X_train.shape[1]} columnas')
print(f'   X_test:  {X_test.shape[0]:,} filas x {X_test.shape[1]} columnas')

print(f'\n3. CHECKLIST DE MODELADO')
print(sep)
checklist = [
    ('Pipeline construido',            True),
    ('Pipeline entrenado con X_train', True),
    ('Predicciones generadas',         True),
    ('Pipeline exportado (.pkl)',       True),
]
for nombre, estado in checklist:
    icono = 'OK' if estado else 'PENDIENTE'
    print(f'   [{icono}]  {nombre}')

print(f'\n4. PROXIMOS PASOS')
print(sep)
print('   -> Continua en el Notebook de Evaluacion')
print('      Abre: Evaluacion.ipynb')
print('      Los objetos pipeline_final, y_pred, y_pred_train estan listos')

print()
print(sep2)
print('  Modelado completado. Pipeline listo para evaluacion.')
print(sep2)


  RESUMEN DE MODELADO
  Modelo: GradientBoostingRegressor

1. PIPELINE
----------------------------------------------------------------------
   preprocesamiento    : ColumnTransformer
   modelo              : GradientBoostingRegressor

2. DATOS
----------------------------------------------------------------------
   X_train: 10,853 filas x 20 columnas
   X_test:  2,714 filas x 20 columnas

3. CHECKLIST DE MODELADO
----------------------------------------------------------------------
   [OK]  Pipeline construido
   [OK]  Pipeline entrenado con X_train
   [OK]  Predicciones generadas
   [OK]  Pipeline exportado (.pkl)

4. PROXIMOS PASOS
----------------------------------------------------------------------
   -> Continua en el Notebook de Evaluacion
      Abre: Evaluacion.ipynb
      Los objetos pipeline_final, y_pred, y_pred_train estan listos

  Modelado completado. Pipeline listo para evaluacion.


---
## Paso 6 — Exportar el Pipeline

El Pipeline entrenado se exporta como `.pkl` usando `joblib`.
Este archivo contiene el preprocesamiento + el modelo entrenado.
Puedes cargarlo en cualquier entorno y predecir sin reentrenar.

> **Esto es lo que se despliega en produccion.**

---

In [12]:
# ── 6. Exportar el Pipeline entrenado ────────────────────────────────────────

os.makedirs('output_lineal', exist_ok=True)

# Guardar el Pipeline
joblib.dump(pipeline_final, 'output_lineal/pipeline_modelo.pkl')
print('Pipeline exportado: output_lineal/pipeline_modelo.pkl')
print(f'Tamano del archivo: {os.path.getsize("output_lineal/pipeline_modelo.pkl") / 1024:.1f} KB')

# Verificar que el pipeline cargado produce las mismas predicciones
pipeline_cargado = joblib.load('output_lineal/pipeline_modelo.pkl')
y_pred_check = pipeline_cargado.predict(X_test[:5])
print(f'\nVerificacion (5 predicciones del pipeline cargado):')
print(y_pred_check.round(4))
print('\nPara usar en produccion:')
print('   pipeline = joblib.load("output_lineal/pipeline_modelo.pkl")')
print('   predicciones = pipeline.predict(nuevos_datos)')

Pipeline exportado: output_lineal/pipeline_modelo.pkl
Tamano del archivo: 142.8 KB

Verificacion (5 predicciones del pipeline cargado):
[12.7339 13.6585 13.8268 13.2094 13.9548]

Para usar en produccion:
   pipeline = joblib.load("output_lineal/pipeline_modelo.pkl")
   predicciones = pipeline.predict(nuevos_datos)
